# Notebook 07 — Orchestrator + ingest + QA with decomposition + gap detection

**Purpose:** Build the §7.1 orchestrator-subagent pattern at both **ingest time** and **query time** — the canonical exam topic for Architecture Patterns. Five parts:

- **A** — Orchestrator basics: dispatch user requests to ingest or QA subagents.
- **B** — Single-question QA: search → read → answer with citations.
- **C** — Query decomposition: split compound questions, run sub-queries in parallel.
- **D** — Knowledge gap detection: structured `KnowledgeGap` instead of fabricated answers.
- **E** — Edge cases: ambiguous routing, degenerate decomposition.

**Exam relevance:** Architecture Patterns (orchestrator-subagent at both ingest and query time).
**Design refs:** §7.1 agent responsibilities, §10A (write), §10C (read), §10I (knowledge gap loop).
**Depends on:** NB 02 (ingest pipeline), NB 05 (cross-source synthesis fixtures), NB 06 (Tool Use mechanics).

**Wiki state:** the 5 committed `SourcePage` fixtures from NB 05 under `data/poc-wiki/sources/` (3 ingest-derived + the contradictory Apollo Q2/Q3 pair).

Subagents run **in-process** as Python functions invoked through the Anthropic Messages API `tools=[...]` loop — same pattern NB 06 settled on after the Claude Agent SDK's `query()` hit the CLI subprocess wall.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import json
import asyncio
import time
from pathlib import Path
from datetime import date

from dotenv import load_dotenv
from anthropic import Anthropic
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.tree import Tree

from engine.tools import search, read_page, find_related, upsert_page, open_pr
from engine.agents.qa import (
    qa_single_question, qa_with_decomposition, qa_with_gap_detection,
    QaAnswer, KnowledgeGap, decompose_query,
)
from engine.agents.orchestrator import run_orchestrator
from engine.models.wiki_config import MarginaliaConfig

console = Console()


In [ ]:
load_dotenv()
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not in env"

client = Anthropic()
config = MarginaliaConfig.load(Path("data/poc-wiki"))
SOURCES = Path("data/poc-wiki").resolve()  # wiki root that holds sources/, etc.

# Smoke: search "Apollo" should return the 2 Apollo pages + the launch-blockers source.
hits = search("Apollo", wiki_root=SOURCES)
print(f"smoke: found {len(hits)} Apollo hits")
for h in hits[:5]:
    print(f"  [{h.score:.3f}] {h.path}  -- {h.title}")


## Part A — Orchestrator basics

A small Sonnet 4.6 agent that routes one user request to either the
ingest subagent (path/URL) or the QA subagent (question). Subagents
are Python functions, invoked through the Messages API tool_use
protocol — same pattern as NB 06.

```
orchestrator (Sonnet 4.6)
├── spawn_ingest_agent  → extract → analyze → synthesize → upsert
└── spawn_qa_agent      → search → read → answer (with gap detection)
```

### Cell 4 — Orchestrator on a write request

The orchestrator decides "this is an ingest task" and dispatches to
`spawn_ingest_agent`, which runs the full NB 02/04 pipeline and writes
the result to a fresh tmp wiki root (so we don't pollute the committed
fixtures).

In [ ]:
tmp_wiki = Path("/tmp/marginalia-nb07-write").resolve()
if tmp_wiki.exists():
    import shutil
    shutil.rmtree(tmp_wiki)
tmp_wiki.mkdir(parents=True)

# Use the existing committed PDF as the input.
pdf_path = Path("data/poc-wiki/raw/clean.pdf").resolve()

t0 = time.monotonic()
write_run = await run_orchestrator(
    f"Please ingest the file at {pdf_path}",
    wiki_root=tmp_wiki,
    config=config,
    client=client,
)
write_wall = time.monotonic() - t0

print(f"orchestrator final answer: {write_run.final_text!r}")
print(f"subagent calls           : {len(write_run.subagent_calls)}")
print(f"orchestrator tokens      : {write_run.orchestrator_tokens_in}/{write_run.orchestrator_tokens_out}")
print(f"orchestrator cost        : ${write_run.orchestrator_cost_usd:.6f}")
print(f"total cost               : ${write_run.total_cost_usd:.6f}")
print(f"wall time                : {write_wall:.2f}s")
print()
print("--- subagent calls ---")
for call in write_run.subagent_calls:
    print(f"  {call.agent}({call.input}) → {call.output_summary}")

# Confirm a page got written.
written = sorted(tmp_wiki.rglob("*.md"))
print(f"\npages written to tmp wiki: {len(written)}")
for w in written:
    print(f"  {w.relative_to(tmp_wiki)}")


### Cell 5 — Tree-render the agent topology

Visual confirmation: orchestrator → ingest subagent → analyze + synthesize.
Subagents share a process; the topology is logical, not OS-level.

In [ ]:
tree = Tree(f"[bold]orchestrator[/bold] (Sonnet 4.6)\n  user: {write_run.user_input!r}\n  final: {write_run.final_text!r}")
for call in write_run.subagent_calls:
    sub = tree.add(f"[cyan]{call.agent}[/cyan]({call.input.get('input') or call.input.get('question')!r})")
    sub.add(f"output: {call.output_summary}")
    if call.tokens_in or call.tokens_out:
        sub.add(f"tokens: {call.tokens_in}/{call.tokens_out}, cost ${call.cost_usd:.6f}")

console.print(tree)


### Cell 6 — Token attribution

The orchestrator should be cheap (small system prompt, dispatch only).
Subagents are where token spend lives — confirms the architecture.

In [ ]:
attribution = Table(title="Per-agent token + cost attribution (write path)")
attribution.add_column("agent")
attribution.add_column("tokens (in/out)", justify="right")
attribution.add_column("cost (USD)", justify="right")
attribution.add_row(
    "orchestrator",
    f"{write_run.orchestrator_tokens_in}/{write_run.orchestrator_tokens_out}",
    f"${write_run.orchestrator_cost_usd:.6f}",
)
for call in write_run.subagent_calls:
    if call.tokens_in or call.tokens_out:
        attribution.add_row(
            call.agent,
            f"{call.tokens_in}/{call.tokens_out}",
            f"${call.cost_usd:.6f}",
        )
attribution.add_section()
attribution.add_row("[bold]TOTAL[/bold]", "—", f"[bold]${write_run.total_cost_usd:.6f}[/bold]")
console.print(attribution)


## Part B — Single-question QA

`qa_single_question(question, wiki_root, client)`: search top-k → read each
hit → ask Sonnet to answer using only the retrieved pages, citing
each via `[[wikilink]]`. Citation integrity is verified against the
retrieved set — the model can't invent paths.

In [ ]:
t0 = time.monotonic()
pricing_answer = await qa_single_question(
    "what did we decide about Q2 launch and pricing for Apollo?",
    wiki_root=SOURCES,
    client=client,
)
qa_wall = time.monotonic() - t0

console.print(Panel(pricing_answer.answer, title=pricing_answer.question, border_style="green"))
print()
print(f"retrieved paths : {pricing_answer.retrieved_paths}")
print(f"citations       : {pricing_answer.citations}")
print(f"dangling        : {pricing_answer.dangling_citations}")
print(f"tokens          : {pricing_answer.tokens_in}/{pricing_answer.tokens_out}")
print(f"cost            : ${pricing_answer.cost_usd:.6f}")
print(f"wall            : {qa_wall:.2f}s")


## Part C — Query decomposition

For compound questions ("X AND Y"), the QA agent first decomposes
into focused sub-queries, runs them in parallel via `asyncio.gather`,
and merges the results. Falls back to single-question on parse
failure or single-element decomposition.

In [ ]:
compound_q = "what did we decide about the Apollo Q2 launch AND was it postponed?"

t0 = time.monotonic()
sub_queries = await decompose_query(compound_q, client=client)
print(f"decomposed into {len(sub_queries)} sub-queries:")
for sq in sub_queries:
    print(f"  [{sq.priority}] {sq.text}")

decomp_t0 = time.monotonic()
decomp_answer = await qa_with_decomposition(
    compound_q,
    wiki_root=SOURCES,
    client=client,
)
decomp_wall = time.monotonic() - decomp_t0

console.print(
    Panel(
        decomp_answer.answer[:1200] + ("…" if len(decomp_answer.answer) > 1200 else ""),
        title=f"decomposed answer ({len(sub_queries)} sub-queries)",
        border_style="cyan",
    )
)
print()
print(f"merged citations : {decomp_answer.citations}")
print(f"total tokens     : {decomp_answer.tokens_in}/{decomp_answer.tokens_out}")
print(f"total cost       : ${decomp_answer.cost_usd:.6f}")
print(f"wall             : {decomp_wall:.2f}s")


### Cell 8b — A/B vs single-question on the same compound question

Run the same compound question through `qa_single_question` (no
decomposition). Compare quality (eyeball), latency, cost. Decomposition
trades one extra LLM call (the split) for parallel sub-retrievals,
which can pay off on truly compound questions but is overhead on
focused ones.

In [ ]:
ab_t0 = time.monotonic()
single_path_answer = await qa_single_question(
    compound_q,
    wiki_root=SOURCES,
    client=client,
)
ab_wall = time.monotonic() - ab_t0

ab = Table(title="A/B: same compound question, two paths")
ab.add_column("path")
ab.add_column("tokens (in/out)", justify="right")
ab.add_column("cost (USD)", justify="right")
ab.add_column("wall", justify="right")
ab.add_column("citations", justify="right")

ab.add_row(
    "single-question",
    f"{single_path_answer.tokens_in}/{single_path_answer.tokens_out}",
    f"${single_path_answer.cost_usd:.6f}",
    f"{ab_wall:.2f}s",
    str(len(single_path_answer.citations)),
)
ab.add_row(
    f"decomposition ({len(sub_queries)} subs)",
    f"{decomp_answer.tokens_in}/{decomp_answer.tokens_out}",
    f"${decomp_answer.cost_usd:.6f}",
    f"{decomp_wall:.2f}s",
    str(len(decomp_answer.citations)),
)
console.print(ab)


## Part D — Knowledge gap detection

When retrieval is thin (fewer than `min_pages` hits or top score
below `min_score`), `qa_with_gap_detection` returns a structured
`KnowledgeGap` instead of fabricating an answer. The signal carries
suggested next ingests so the curator can `marginalia add <URL>`
and re-query.

In [ ]:
gap_t0 = time.monotonic()
gap = await qa_with_gap_detection(
    "what's our exposure to the EU AI Act?",
    wiki_root=SOURCES,
    client=client,
    min_pages=2,
    min_score=0.05,
)
gap_wall = time.monotonic() - gap_t0

print(f"result type: {type(gap).__name__}")
print()
if isinstance(gap, KnowledgeGap):
    panel_body = (
        f"[bold]reason:[/bold] {gap.reason}\n\n"
        f"[bold]retrieved:[/bold] {gap.retrieved_paths or '(none)'}\n\n"
        f"[bold]suggested ingests:[/bold]\n"
        + "\n".join(f"  • {s}" for s in gap.suggested_ingests)
    )
    console.print(Panel(panel_body, title="KnowledgeGap", border_style="yellow"))
    print()
    print(f"cost: ${gap.cost_usd:.6f}, wall: {gap_wall:.2f}s")
else:
    console.print(Panel(gap.answer[:600], title="answered (unexpected)", border_style="green"))


### Cell 9b — Close the loop: ingest a suggested source, re-query

Simulate `marginalia add <suggested URL>` by hand-writing a SourcePage
about the EU AI Act into a tmp wiki copy that includes the original
fixtures, then re-query.

In [ ]:
# Build a tmp wiki = committed sources + a hand-curated EU AI Act page.
import shutil

loop_wiki = Path("/tmp/marginalia-nb07-loop").resolve()
if loop_wiki.exists():
    shutil.rmtree(loop_wiki)
loop_wiki.mkdir(parents=True)

# Copy committed sources/.
shutil.copytree(SOURCES / "sources", loop_wiki / "sources")

# Hand-curated EU AI Act SourcePage — simulates a successful `marginalia add`.
eu_ai_fm = {
    "title": "EU AI Act — overview",
    "type": "source",
    "status": "active",
    "created": "2026-04-15",
    "last_synced": "2026-05-03",
    "sources": [{
        "ref": "artificialintelligenceact.eu",
        "kind": "web_page",
        "captured": "2026-05-03",
        "authority": "canonical",
    }],
    "tags": ["compliance", "eu", "ai-act"],
}
eu_ai_body = (
    "# EU AI Act overview\n\n"
    "The EU AI Act classifies AI systems into risk tiers (unacceptable, high, "
    "limited, minimal). Apollo and other Marginalia products may fall under "
    "high-risk if they're used for hiring decisions, critical infrastructure, "
    "or biometric ID. Compliance obligations include data governance, "
    "transparency disclosures, and conformity assessments. The Act enters "
    "force in stages between 2025 and 2027.\n"
)
upsert_page(
    "sources/eu-ai-act-overview",
    eu_ai_fm,
    eu_ai_body,
    wiki_root=loop_wiki,
)
print("hand-wrote sources/eu-ai-act-overview.md to loop wiki")

# Re-query.
loop_t0 = time.monotonic()
re_query_result = await qa_with_gap_detection(
    "what's our exposure to the EU AI Act?",
    wiki_root=loop_wiki,
    client=client,
    min_pages=1,  # one page is now enough
    min_score=0.05,
)
loop_wall = time.monotonic() - loop_t0
print(f"re-query result type: {type(re_query_result).__name__}")
if isinstance(re_query_result, QaAnswer):
    console.print(
        Panel(
            re_query_result.answer,
            title="answered after gap closure",
            border_style="green",
        )
    )
    print(f"citations: {re_query_result.citations}")
    print(f"cost: ${re_query_result.cost_usd:.6f}, wall: {loop_wall:.2f}s")


## Part E — Edge cases

Two corners of the orchestrator + QA contracts:

1. **Ambiguous input.** "tell me about pricing" — no URL/path, but
   not phrased as a question either. The orchestrator routes to QA
   by default (gap detection signals if coverage is thin).
2. **Degenerate decomposition.** A non-compound question goes
   through `qa_with_decomposition`; `decompose_query` returns one
   item, the merge step is a passthrough.

In [ ]:
# 1. Ambiguous input.
amb_t0 = time.monotonic()
amb_run = await run_orchestrator(
    "tell me about pricing",
    wiki_root=SOURCES,
    config=config,
    client=client,
)
amb_wall = time.monotonic() - amb_t0

print("--- ambiguous input ---")
print(f"final answer: {amb_run.final_text}")
print(f"subagent calls: {[c.agent for c in amb_run.subagent_calls]}")

# 2. Degenerate decomposition: a focused single question.
deg_q = "when was the Apollo Q2 launch confirmed?"
deg_t0 = time.monotonic()
deg_subs = await decompose_query(deg_q, client=client)
deg_answer = await qa_with_decomposition(deg_q, wiki_root=SOURCES, client=client)
deg_wall = time.monotonic() - deg_t0

print()
print("--- degenerate decomposition ---")
print(f"sub-queries: {len(deg_subs)} ({[s.text for s in deg_subs]})")
print(f"answer body has {'NO ' if 'Sub-question' not in deg_answer.answer else ''}sub-question headers (passthrough={deg_answer.answer.count('Sub-question') == 0})")
print(f"citations: {deg_answer.citations}")
print(f"cost: ${deg_answer.cost_usd:.6f}, wall: {deg_wall:.2f}s")


### Cell 11 — Receipts: write `engine/decisions/orchestrator-and-qa.md`

In [ ]:
DECISIONS_PATH = Path("../engine/decisions/orchestrator-and-qa.md")
DECISIONS_PATH.parent.mkdir(parents=True, exist_ok=True)

# Decompose A/B numbers.
single_cost = single_path_answer.cost_usd
decomp_cost = decomp_answer.cost_usd

# Gap demo numbers.
gap_path_cost = (gap.cost_usd if isinstance(gap, KnowledgeGap) else 0.0)
re_query_cost = (re_query_result.cost_usd if isinstance(re_query_result, QaAnswer) else 0.0)

receipts = f"""# Orchestrator + QA receipts

**Last verified:** {date.today().isoformat()}
**Generated by:** `notebooks/07_orchestrator_qa.ipynb`
**Design refs:** `docs/marginalia-design.md` §7.1 (orchestrator + QA agent), §10A (write scenario), §10C (read scenario), §10I (knowledge gap loop).

## Architecture

```
orchestrator (Sonnet 4.6) — dispatch + composition
├── spawn_ingest_agent → extract → analyze (Haiku 4.5) → synthesize (Sonnet 4.6) → upsert
└── spawn_qa_agent     → qa_with_gap_detection (Sonnet 4.6)
                        ├── thin retrieval → KnowledgeGap signal
                        └── adequate retrieval → qa_single_question
```

Subagents run in-process via the Anthropic Messages API tool_use loop —
NB 06 verified that the Claude Agent SDK's `query()` requires a
`claude` CLI subprocess and fails when nested inside another claude
session. The Messages API path is more transparent anyway: the agent
loop is one `messages.create` per turn, with `tool_use` blocks
dispatched as Python function calls.

## Per-agent token attribution (write path)

| Agent | Tokens (in/out) | Cost (USD) |
|---|---|---|
| orchestrator | {write_run.orchestrator_tokens_in}/{write_run.orchestrator_tokens_out} | ${write_run.orchestrator_cost_usd:.6f} |
"""
for call in write_run.subagent_calls:
    if call.tokens_in or call.tokens_out:
        receipts += f"| {call.agent} | {call.tokens_in}/{call.tokens_out} | ${call.cost_usd:.6f} |\n"
receipts += f"| **TOTAL** | — | **${write_run.total_cost_usd:.6f}** |\n"

receipts += f"""

The orchestrator stays cheap (~{write_run.orchestrator_tokens_in + write_run.orchestrator_tokens_out} tokens
total) — it just dispatches. Subagents do the heavy work.

## QA paths

| Path | Tokens (in/out) | Cost (USD) | Wall |
|---|---|---|---|
| Single-question on compound | {single_path_answer.tokens_in}/{single_path_answer.tokens_out} | ${single_cost:.6f} | {ab_wall:.2f}s |
| Decomposition ({len(sub_queries)} subs) on compound | {decomp_answer.tokens_in}/{decomp_answer.tokens_out} | ${decomp_cost:.6f} | {decomp_wall:.2f}s |

Decomposition pays an extra Sonnet call for the split. The win is parallel
sub-retrievals via `asyncio.gather` — pays off when sub-queries hit
disjoint page neighborhoods. On focused questions, decomposition is
overhead.

## Knowledge gap loop

Question: "what's our exposure to the EU AI Act?" against the 5-page wiki.

- Initial query: returned **{type(gap).__name__}** with {len(gap.suggested_ingests) if isinstance(gap, KnowledgeGap) else 0} suggested ingests.
- Suggested ingests included specific URLs and `search:` hints.
- Cost of the gap signal: ${gap_path_cost:.6f}.
- After hand-curating an `EU AI Act overview` SourcePage and re-querying:
  returned **{type(re_query_result).__name__}** with {len(re_query_result.citations) if isinstance(re_query_result, QaAnswer) else 0} citations, cost ${re_query_cost:.6f}.

The loop closed: gap signal → ingest → answer. The §10I scenario works.

## Selected design choices

| Decision | Choice | Rationale |
|---|---|---|
| Subagent execution | In-process Python via Messages API tool_use | NB 06 verified Agent SDK `query()` requires CLI subprocess; nested fails. |
| Search backend | Naive TF-IDF over `*.md` files | 5–7 page PoC; signature stable for vector swap. |
| `wiki_root` parameter | Explicit on every tool function | Tests use `tmp_path`; notebook uses tmp dir for write demos to avoid polluting committed fixtures. |
| `open_pr` | Print-stub returning `PullRequest` metadata | Real `gh` integration is design §9.6 territory. |
| Citation integrity | `dangling_citations` field on `QaAnswer` | Same retry-error-shape convention as NB 05 cross-source synthesis. |
| Gap threshold | `min_pages=2`, `min_score=0.1` defaults | Tunable per-call; PoC sweet spot on the 5-page wiki. |

## Caveats

- 5-page wiki is small enough that retrieval recall isn't a meaningful
  signal — production wikis with 100+ pages will surface different
  failure modes.
- The decomposition prompt asks for 2–4 sub-queries; for genuinely
  multi-part questions it sometimes splits when one would have
  sufficed. Tunable via the prompt; receipt that as priors for NB 11.
- `open_pr` doesn't open a real PR. The `PullRequest` shape is stable
  for when `gh` integration lands.

## Re-running

```python
from pathlib import Path
from anthropic import Anthropic
from engine.agents.qa import qa_with_gap_detection
from engine.models.wiki_config import MarginaliaConfig

client = Anthropic()
config = MarginaliaConfig.load(Path("notebooks/data/poc-wiki"))
result = await qa_with_gap_detection(
    "what about Apollo?",
    wiki_root=Path("notebooks/data/poc-wiki"),
    client=client,
)
```
"""

DECISIONS_PATH.write_text(receipts, encoding="utf-8")
print(f"wrote {DECISIONS_PATH.resolve()}  ({DECISIONS_PATH.stat().st_size} bytes)")


## What to extract

| Notebook artifact | Extracts to |
|---|---|
| `run_orchestrator()` + Pydantic types | `engine/agents/orchestrator/main.py` (extracted) |
| `qa_single_question()` + `QaAnswer` | `engine/agents/qa/single_question.py` (extracted) |
| `decompose_query()` + `qa_with_decomposition()` + `SubQuery` | `engine/agents/qa/decompose.py` (extracted) |
| `qa_with_gap_detection()` + `KnowledgeGap` | `engine/agents/qa/gap_detection.py` (extracted) |
| `marginalia.search` / `read_page` / `find_related` / `upsert_page` / `open_pr` | `engine/tools/*.py` (extracted) |
| Versioned prompts | `engine/prompts/{orchestrator,qa_single,qa_decompose,qa_gap_signal}.md` (extracted) |
| Cell 11 receipts | `engine/decisions/orchestrator-and-qa.md` (extracted) |
| Agent topology overview | `engine/agents/README.md` (extracted) |

**Notebook-only (intentionally not extracted):**
- The tree-render trace visualization (cell 5) — diagnostic only.
- The token-attribution table (cell 6) — receipts version lives in `orchestrator-and-qa.md`.
- The hand-curated EU AI Act SourcePage (cell 9b) — single-use loop demo.
- Tmp wiki management (`/tmp/marginalia-nb07-*`) — notebook plumbing.

**Out of scope (deferred):**
- `engine/tools/lint_check.py` — NB 08.
- Real `gh` GitHub PR integration — `open_pr` is a metadata-only stub.
- CLI verbs (`marginalia ask`, `marginalia ingest`) — wiring agents/tools into Typer is a separate concern.
- Atomic multi-file PRs (per §10A step 6) — the synthesis agent (NB 05) does the multi-file reasoning; full atomic-PR plumbing comes when the GH integration lands.
- Subagent timeouts + cancellation — production concern, deferred.

**`CACHE_VERSION` discipline:** not triggered (no edits to existing prompts). Four new prompts (`orchestrator.md`, `qa_single.md`, `qa_decompose.md`, `qa_gap_signal.md`) start at v1; future edits bump in lockstep with `CACHE_VERSION` once it lands in NB 10.
